# SVD·최소제곱·PCA

행렬 분해로 복원·최소제곱해·차원축소가 연결되는 원리 확인

코드 셀을 위에서 아래로 실행하고 문제의 조건을 바꾸어 결과 비교

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.set_printoptions(precision=5, suppress=True)

## SVD로 복원

A=[[2,-1],[-1,2]]를 U·Σ·Vᵀ로 분해하시오. 전체 복원과 특잇값 하나만 사용한 복원의 차이를 비교하시오.

In [ ]:
A = np.array([[2.,-1.],[-1.,2.]])
U,s,Vt = np.linalg.svd(A)
print('특잇값:',s)
print('복원:',U@np.diag(s)@Vt)
rank1 = s[0]*np.outer(U[:,0],Vt[0])
print('계수 1 근사:',rank1)
assert np.allclose(U@np.diag(s)@Vt,A)
assert np.isclose(np.linalg.norm(A-rank1),s[1])
print('Ax=[0,3]의 해:',np.linalg.solve(A,[0,3]))

## 의사역행렬로 최소제곱해 구하기

A=[[-1,1],[0,1],[0,1]], b=[0,1,3]이다. 최소제곱해와 잔차를 구하고 잔차가 A의 열에 수직인지 확인하시오.

In [ ]:
A=np.array([[-1.,1.],[0.,1.],[0.,1.]])
b=np.array([0.,1.,3.])
solution=np.linalg.pinv(A)@b
residual=b-A@solution
print("해:",solution,"잔차:",residual)
print("A.T @ 잔차:",A.T@residual)

## 중심을 옮긴 뒤 PCA

다섯 점 `[170,70],[150,45],[160,55],[180,60],[172,80]`을 중심화한 뒤 첫 주성분에 투영하시오. 공분산의 고유벡터와 SVD 방향을 비교하시오.

In [ ]:
X = np.array([[170.,70.],[150.,45.],[160.,55.],[180.,60.],[172.,80.]])
mean = X.mean(axis=0); Z = X-mean
C = Z.T@Z/len(Z)  # 모집단 분산 기준; 표본 기준과 방향은 같음
lam,V = np.linalg.eigh(C); pc = V[:,-1]
_,singular,Vt = np.linalg.svd(Z,full_matrices=False)
score = Z@pc
reconstructed = np.outer(score,pc)+mean
print('평균:',mean,'공분산:',C,'분산 비율:',lam[-1]/lam.sum())
assert np.allclose(np.outer(pc,pc),np.outer(Vt[0],Vt[0]))
assert np.allclose(lam, np.sort(singular**2/len(Z)))
plt.scatter(*X.T,label='Data');plt.scatter(*reconstructed.T,label='PC1 reconstruction')
for a,b in zip(X,reconstructed):plt.plot([a[0],b[0]],[a[1],b[1]],'k--',alpha=.3)
plt.axis('equal');plt.legend();plt.show()

고유벡터의 부호가 반대여도 같은 축. 좌표가 아닌 투영·복원 결과로 비교.

자료 전체에 [100,200]을 더한 뒤 중심화를 다시 적용하시오. 주성분 방향이 유지되는지 확인하시오.